In [1]:
# -*- coding: utf-8 -*-
"""
АХО-КОРАСИК + СИСТЕМА ШТРАФОВ
Простая реализация для поиска нескольких паттернов за один проход
Google Colab версия
"""

class AhoCorasik:
    """Простая реализация алгоритма Ахо-Корасик"""

    def __init__(self):
        # Каждый узел: {буква: индекс_следующего_узла}
        self.trie = [{}]  # дерево (бор)
        self.fail = [0]   # провальные ссылки (куда идти при ошибке)
        self.output = [[]]  # какие паттерны заканчиваются в этом узле

    def add_pattern(self, pattern, pattern_name):
        """
        Добавить паттерн в автомат
        pattern: строка для поиска (например, "СНИЛС")
        pattern_name: название (например, "СНИЛС_123")
        """
        node = 0
        # Идём по дереву, создаём новые узлы если нужно
        for char in pattern:
            if char not in self.trie[node]:
                self.trie[node][char] = len(self.trie)
                self.trie.append({})
                self.fail.append(0)
                self.output.append([])
            node = self.trie[node][char]
        # В конце пути сохраняем название паттерна
        self.output[node].append(pattern_name)

    def build_automaton(self):
        """Построить провальные ссылки (самая важная часть!)"""
        from collections import deque
        queue = deque()

        # Инициализация: для всех букв из корня
        for char, next_node in self.trie[0].items():
            self.fail[next_node] = 0
            queue.append(next_node)

        # BFS по дереву
        while queue:
            current = queue.popleft()

            # Для каждой буквы из текущего узла
            for char, next_node in self.trie[current].items():
                queue.append(next_node)

                # Ищем провальную ссылку
                f = self.fail[current]
                while f and char not in self.trie[f]:
                    f = self.fail[f]

                if char in self.trie[f]:
                    self.fail[next_node] = self.trie[f][char]
                else:
                    self.fail[next_node] = 0

                # Копируем выходы из провальной ссылки
                self.output[next_node].extend(self.output[self.fail[next_node]])

    def search(self, text):
        """
        Поиск всех паттернов в тексте
        Возвращает список словарей: {позиция: [найденные_паттерны]}
        """
        results = []
        node = 0

        for i, char in enumerate(text):
            # Идём по провальным ссылкам пока не найдём букву
            while node and char not in self.trie[node]:
                node = self.fail[node]

            if char in self.trie[node]:
                node = self.trie[node][char]
            else:
                node = 0

            # Если в этом узле есть паттерны — записываем
            if self.output[node]:
                results.append({
                    'position': i,
                    'patterns': self.output[node].copy()
                })

        return results


# ==================== СОЗДАЁМ ТЕКСТОВЫЕ ДАННЫЕ ====================

text = """
СЛУЖЕБНАЯ ЗАПИСКА

Кому: Начальнику отдела безопасности
От: Иванова Ивана Ивановича

Уважаемый Пётр Петрович!

Направляю Вам личные данные сотрудника Петрова Петра Петровича:
- Паспорт: 4510 123456
- СНИЛС: 123-456-789 01
- ИНН: 123456789012
- Телефон: +7-903-123-45-67
- Email: petrov@company.ru

Также в документе упоминается другой сотрудник: Сидорова Анна Сергеевна
Её паспорт: 4600 987654

С уважением, Иванов И.И.

P.S. Документ имеет гриф "КОНФИДЕНЦИАЛЬНО"
"""

print("="*70)
print("ТЕКСТ ДЛЯ АНАЛИЗА:")
print("="*70)
print(text)
print("="*70)


# ==================== НАСТРАИВАЕМ ПАТТЕРНЫ И ШТРАФЫ ====================

# Словарь: название паттерна -> строка для поиска
patterns = {
    "Имя_Иванов": "Иванов",
    "Имя_Петров": "Петров",
    "Имя_Сидорова": "Сидорова",
    "Паспорт_4510": "4510 123456",
    "Паспорт_4600": "4600 987654",
    "СНИЛС": "123-456-789 01",
    "ИНН": "123456789012",
    "Телефон": "+7-903-123-45-67",
    "Email": "@company.ru",
    "Гриф_секретно": "КОНФИДЕНЦИАЛЬНО"
}

# Система штрафов (пенальти) для каждого типа данных
penalties = {
    "Имя_Иванов": 1,
    "Имя_Петров": 1,
    "Имя_Сидорова": 1,
    "Паспорт_4510": 8,
    "Паспорт_4600": 8,
    "СНИЛС": 10,
    "ИНН": 5,
    "Телефон": 3,
    "Email": 2,
    "Гриф_секретно": 20  # Очень большой штраф!
}

# ==================== ЗАПУСКАЕМ АХО-КОРАСИК ====================

# Создаём автомат
automaton = AhoCorasik()

# Добавляем все паттерны
for name, pattern in patterns.items():
    automaton.add_pattern(pattern, name)
    print(f"Добавлен паттерн: {name} = '{pattern}' (штраф: {penalties[name]})")

# Строим провальные ссылки
automaton.build_automaton()
print("\nАвтомат построен! Начинаем поиск...\n")

# Ищем все совпадения
matches = automaton.search(text)

# ==================== ОБРАБАТЫВАЕМ РЕЗУЛЬТАТЫ ====================

# Собираем уникальные найденные паттерны и считаем штраф
found_patterns = {}
total_penalty = 0

for match in matches:
    pos = match['position']
    for pattern_name in match['patterns']:
        if pattern_name not in found_patterns:
            found_patterns[pattern_name] = []
        # Запоминаем позицию (можно несколько)
        if pos not in found_patterns[pattern_name]:
            found_patterns[pattern_name].append(pos)

print("="*70)
print("РЕЗУЛЬТАТЫ ПОИСКА:")
print("="*70)

# Выводим все найденные паттерны
for name in sorted(found_patterns.keys()):
    penalty = penalties[name]
    positions = found_patterns[name]
    total_penalty += penalty
    print(f"✓ {name}: штраф {penalty} — найден на позициях {positions}")

print("\n" + "="*70)
print(f"БАЗОВЫЙ ШТРАФ (сумма): {total_penalty}")
print("="*70)

# ==================== КОНТЕКСТНЫЕ МОДИФИКАТОРЫ ====================

context_multiplier = 1.0
reasons = []

# 1. Проверяем наличие грифа "секретно"
if "Гриф_секретно" in found_patterns:
    context_multiplier *= 3
    reasons.append("найден гриф 'КОНФИДЕНЦИАЛЬНО' (×3)")

# 2. Проверяем комбинацию паспорт + СНИЛС (очень опасно!)
has_passport = any(p.startswith("Паспорт") for p in found_patterns)
has_snils = "СНИЛС" in found_patterns
if has_passport and has_snils:
    context_multiplier *= 2
    reasons.append("паспорт + СНИЛС вместе (×2)")

# 3. Проверяем комбинацию ФИО + паспорт
has_name = any(p.startswith("Имя_") for p in found_patterns)
if has_name and has_passport:
    context_multiplier *= 1.5
    reasons.append("ФИО + паспорт вместе (×1.5)")

# 4. Есть ли email (признак утечки)
if "Email" in found_patterns:
    context_multiplier *= 1.2
    reasons.append("найден email (×1.2)")

# ==================== ИТОГОВЫЙ РАСЧЁТ ====================

final_penalty = total_penalty * context_multiplier

print("\n" + "="*70)
print("КОНТЕКСТНЫЕ МОДИФИКАТОРЫ:")
print("="*70)
if reasons:
    for reason in reasons:
        print(f"  • {reason}")
else:
    print("  • Дополнительных множителей нет")
print(f"\nИтоговый множитель: ×{context_multiplier}")

print("\n" + "="*70)
print("ИТОГОВЫЙ ШТРАФ (с учётом контекста):")
print("="*70)
print(f"  Базовый штраф: {total_penalty}")
print(f"  × Множитель: {context_multiplier}")
print(f"  = {final_penalty:.1f} баллов")
print("="*70)

# ==================== ВЕРДИКТ ====================

print("\n" + "="*70)
print("ВЕРДИКТ:")
print("="*70)

if final_penalty >= 50:
    print("🚨 КРИТИЧЕСКАЯ УГРОЗА! Документ заблокирован.")
    print("   → Уведомить службу безопасности.")
    print("   → Зафиксировать инцидент.")
elif final_penalty >= 20:
    print("⚠️ ВЫСОКИЙ РИСК! Требуется проверка.")
    print("   → Отправить на дополнительную модерацию.")
elif final_penalty >= 10:
    print("⚡ ПОВЫШЕННОЕ ВНИМАНИЕ! Документ помечен.")
    print("   → Логирование события.")
else:
    print("✅ Безопасно. Пропустить.")

print("="*70)

# ==================== ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ ====================

print("\n" + "="*70)
print("КАК ЭТО РАБОТАЕТ (для понимания):")
print("="*70)
print("""
1. Ахо-Корасик за один проход по тексту нашёл ВСЕ паттерны:
   - Имена сотрудников
   - Паспорта, СНИЛС, ИНН
   - Телефон, email
   - Гриф секретности

2. Система штрафов оценила опасность каждого найденного элемента

3. Контекстные множители учли:
   - Наличие грифа 'КОНФИДЕНЦИАЛЬНО' (×3)
   - Опасные комбинации (паспорт + СНИЛС = ×2)
   - Наличие ФИО и паспорта вместе (×1.5)

4. Итоговый штраф = сумма_штрафов × множители

ЧЕМ ВЫШЕ ШТРАФ — ТЕМ ОПАСНЕЕ ДОКУМЕНТ!
""")

ТЕКСТ ДЛЯ АНАЛИЗА:

СЛУЖЕБНАЯ ЗАПИСКА

Кому: Начальнику отдела безопасности
От: Иванова Ивана Ивановича

Уважаемый Пётр Петрович!

Направляю Вам личные данные сотрудника Петрова Петра Петровича:
- Паспорт: 4510 123456
- СНИЛС: 123-456-789 01
- ИНН: 123456789012
- Телефон: +7-903-123-45-67
- Email: petrov@company.ru

Также в документе упоминается другой сотрудник: Сидорова Анна Сергеевна
Её паспорт: 4600 987654

С уважением, Иванов И.И.

P.S. Документ имеет гриф "КОНФИДЕНЦИАЛЬНО"

Добавлен паттерн: Имя_Иванов = 'Иванов' (штраф: 1)
Добавлен паттерн: Имя_Петров = 'Петров' (штраф: 1)
Добавлен паттерн: Имя_Сидорова = 'Сидорова' (штраф: 1)
Добавлен паттерн: Паспорт_4510 = '4510 123456' (штраф: 8)
Добавлен паттерн: Паспорт_4600 = '4600 987654' (штраф: 8)
Добавлен паттерн: СНИЛС = '123-456-789 01' (штраф: 10)
Добавлен паттерн: ИНН = '123456789012' (штраф: 5)
Добавлен паттерн: Телефон = '+7-903-123-45-67' (штраф: 3)
Добавлен паттерн: Email = '@company.ru' (штраф: 2)
Добавлен паттерн: Гриф_секре